In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from chronos import BaseChronosPipeline
from aeon.transformations.collection.base import BaseCollectionTransformer

2026-03-16 10:07:19.193578: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = 'Wine'

In [3]:
# Test all embedding transformers
from time import perf_counter
from tscglue import utils, data_loader

X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 4)
print(f"X_test shape: {X_test.shape}\n")

transformers = [
    ("MantisV2", MantisEmbedding()),
    ("chronos-bolt-tiny", Chronos2Embedding(model_id="amazon/chronos-bolt-tiny")),
    ("chronos-bolt-mini", Chronos2Embedding(model_id="amazon/chronos-bolt-mini")),
    ("chronos-bolt-small", Chronos2Embedding(model_id="amazon/chronos-bolt-small")),
    ("chronos-bolt-base", Chronos2Embedding(model_id="amazon/chronos-bolt-base")),
    ("chronos-2", Chronos2Embedding(model_id="amazon/chronos-2")),
]

for name, t in transformers:
    t0 = perf_counter()
    emb = t.transform(X_test)
    elapsed = perf_counter() - t0
    print(f"{name}: {emb.shape} ({elapsed:.1f}s)")

X_test shape: (54, 1, 234)

MantisV2: (54, 256) (5.0s)
chronos-bolt-tiny: (54, 256) (1.9s)
chronos-bolt-mini: (54, 384) (2.5s)
chronos-bolt-small: (54, 512) (2.6s)
chronos-bolt-base: (54, 768) (4.2s)
chronos-2: (54, 768) (6.9s)


In [4]:
from time import perf_counter
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from sklearn.linear_model import RidgeClassifierCV
from threadpoolctl import threadpool_limits
from tscglue import data_loader

class RidgeClassifierCVDecisionProba(RidgeClassifierCV):
    def fit(self, X, y):
        with threadpool_limits(limits=1):
            return super().fit(X, y)

X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 12)

transformers = [
    ("MantisV2", MantisEmbedding()),
    ("chronos-bolt-tiny", Chronos2Embedding(model_id="amazon/chronos-bolt-tiny")),
    ("chronos-bolt-mini", Chronos2Embedding(model_id="amazon/chronos-bolt-mini")),
    ("chronos-bolt-small", Chronos2Embedding(model_id="amazon/chronos-bolt-small")),
    ("chronos-bolt-base", Chronos2Embedding(model_id="amazon/chronos-bolt-base")),
    ("chronos-2", Chronos2Embedding(model_id="amazon/chronos-2")),
]

for name, t in transformers:
    t0 = perf_counter()
    Xt_train = t.transform(X_train)
    Xt_test = t.transform(X_test)
    clf = make_pipeline(StandardScaler(), RidgeClassifierCVDecisionProba(alphas=np.logspace(-3, 3, 10)))
    clf.fit(Xt_train, y_train)
    y_pred = clf.predict(Xt_test)
    elapsed = perf_counter() - t0
    print(f"{name}: acc={accuracy_score(y_test, y_pred):.4f} ({elapsed:.1f}s)")

MantisV2: acc=0.9074 (8.1s)
chronos-bolt-tiny: acc=0.9074 (3.9s)
chronos-bolt-mini: acc=0.8148 (3.9s)
chronos-bolt-small: acc=0.7778 (5.2s)
chronos-bolt-base: acc=0.9444 (9.7s)
chronos-2: acc=0.9259 (14.1s)
